In [ ]:
from pathlib import Path

import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
from IPython.display import display

from resources import find_rotation_error
from resources.cut_off_edges import cut_off_edges
from resources.find_x import (
    calculate_x_profile,
    find_x_area_center, find_x_area_center_split,
)
from resources.find_y import (
    calculate_y_profile,
    find_y_argmax, find_y_area_center,
)
from resources.find_z import (
    calculate_z_profile,
    find_z_argmax, find_z_area_center,
)
from resources.rotate_image import rotate_volume
from resources.standarize_image import standardize_image
from resources.find_tilt_error import find_tilt_error

In [ ]:
PROJECT_DIR = Path.cwd().parents[2]

DATA_DIR = PROJECT_DIR / "data"
SRC_DIR = PROJECT_DIR / "src"

NIFTI_DIR = PROJECT_DIR / "data" / "niftis"

# UID obrazu do testowania
UID = "493wxa73"

In [ ]:
file_path = NIFTI_DIR / f"{UID}.nii.gz"

image = nib.load(file_path)
# Obróbka obrazu; funkcja zwraca obiekt NIfTI
processed_image = cut_off_edges(image)
processed_image = cut_off_edges(processed_image,x_left=35,x_right=35,y_front=40,y_back=40,z_down=30,z_up=30)
# Standaryzacja obrazu
processed_image = standardize_image(processed_image, percent_top=0.02)

volume = processed_image.get_fdata(dtype=np.float32)

x_size, y_size, z_size = volume.shape

print("UID:", UID)
print("Shape:", volume.shape)
print("Voxel size:", image.header.get_zooms()[:3])
print("Orientation:", nib.aff2axcodes(image.affine))

In [ ]:

x_profile = calculate_x_profile(volume)
y_profile = calculate_y_profile(volume)
z_profile = calculate_z_profile(volume)

x_centre = int(find_x_area_center(x_profile))
y_centre = int(find_y_area_center(y_profile))
z_centre = int(find_z_area_center(z_profile))

xyz_centre = (
    x_centre,
    y_centre,
    z_centre,
)

print()
print("Wyznaczony środek:")
print("X:", x_centre)
print("Y:", y_centre)
print("Z:", z_centre)

In [ ]:
tilt_error = find_tilt_error(processed_image)
tilt_error